# Backpropagate through a network by hand

Implement scalar and batched gradients without an autodiff library. Use a stable classifier loss and compare analytic and numerical derivatives.

Fill in each starter, then run its check cell. Restart the kernel and run all cells before submitting. Keep your interpretation in the last Markdown cell.


## 1. Backpropagate two paths into one node

Add two paths through a shared input.

Implement shared_path(x,y) for z=x+y and loss=z*z+z. Return (loss,(derivative with respect to x, derivative with respect to y)).


In [ ]:
def shared_path(x,y):
    raise NotImplementedError


In [ ]:
assert shared_path(1,2) == (12,(7,7))
assert shared_path(-1,1) == (0,(1,1))
assert shared_path(-3,1) == (2,(-3,-3))
assert shared_path(0,0) == (0,(1,1))
print("OK")


## 2. Backpropagate a batched dense layer

Propagate gradients through a dense layer.

Implement dense_backward(x,w,g) with rectangular compatible nonempty lists. Given upstream derivative g, return (dx,dw,db) with the same nested-list shapes as x, w and bias. Do not add an averaging factor.


In [ ]:
def dense_backward(x,w,g):
    raise NotImplementedError


In [ ]:
assert dense_backward([[2],[3]],[[4,5]],[[1,0],[0,1]]) == ([[4],[5]],[[2,3]],[1,1])
assert dense_backward([[1,2]],[[5],[7]],[[3]]) == ([[15,21]],[[3],[6]],[3])
assert dense_backward([[0],[1]],[[2]],[[1],[-1]]) == ([[2],[-2]],[[-1]],[0])
print("OK")


## 3. Compute stable softmax loss and gradient

Calculate logit loss and derivatives.

Implement softmax_loss_gradient(logits,label) for finite logits and an integer class index. Reject an empty list or invalid target, including bool. Return (loss, gradient list) for one example.


In [ ]:
def softmax_loss_gradient(logits,label):
    raise NotImplementedError


In [ ]:
from math import log
loss,grad = softmax_loss_gradient([1000,1000],1)
assert abs(loss-log(2)) < 1e-12 and grad == [0.5,-0.5]
loss,grad = softmax_loss_gradient([0,-1000],1)
assert (loss,grad) == (1000,[1,-1])
loss,grad = softmax_loss_gradient([5],0)
assert (loss,grad) == (0,[0])
for scores,label in (([],0),([1],1),([1],True),([1,2],-1)):
    try:
        softmax_loss_gradient(scores,label)
    except ValueError:
        pass
    else:
        raise AssertionError("reject invalid logits or target")
print("OK")

## 4. Estimate derivatives one coordinate at a time

Check an analytic gradient with numerical differences.

Implement numerical_gradient(objective,values,step) for a scalar callable and list of finite values. Step is a finite positive size or an invalid nonpositive value to reject. Use symmetric perturbations and preserve the original list.


In [ ]:
def numerical_gradient(objective,values,step):
    raise NotImplementedError


In [ ]:
values = [2.,3.]
grad = numerical_gradient(lambda v:v[0]**2+v[1]**3,values,1e-4)
assert len(grad) == 2 and all(abs(a-b)<1e-5 for a,b in zip(grad,[4,27]))
assert values == [2.,3.]
assert numerical_gradient(sum,[],1e-4) == []
assert abs(numerical_gradient(lambda v:7*v[0],[0.],1e-4)[0]-7) < 1e-12
point = [1.5,-2.0,0.25]
grad = numerical_gradient(lambda v:v[0]*v[1]+v[2]**2,point,1e-4)
assert len(grad) == 3 and all(abs(a-b)<1e-8 for a,b in zip(grad,[-2,1.5,0.5]))
for bad in (0,-1e-4):
    try:
        numerical_gradient(sum,[2.],bad)
    except ValueError:
        pass
    else:
        raise AssertionError("reject nonpositive step")
print("OK")

## 5. Check the backward pass of a two-layer network

Implement network_loss_gradient(x,labels,params). Use a 2→2 tanh hidden layer and a 2→2 logit layer.
params holds four row-major W1 values (W1[d][h], inputs as rows, so Z = XW + b), two b1 values, four row-major W2 values, then two b2 values.
Compute mean cross-entropy and return (mean loss, flattened gradient in the same layout). Reuse
softmax_loss_gradient and dense_backward. The derivative through tanh(h) is 1 minus the square
of its output. The check measures every coordinate of the same loss function with finite differences.


In [ ]:
def network_loss_gradient(x,labels,params):
    raise NotImplementedError


In [ ]:
x,labels = [[0.2,-0.7],[1.1,0.3],[0.0,0.4]],[1,0,1]
p = [0.3,-0.2,0.4,0.1,0.1,0.,-0.4,0.8,0.4,0.2,-0.1,0.2]
loss,gradient = network_loss_gradient(x,labels,p)
assert abs(loss - 0.7715891479696142) < 1e-9
numerical = numerical_gradient(lambda values: network_loss_gradient(x,labels,values)[0],p,1e-5)
assert loss > 0 and len(gradient) == 12
assert max(abs(g-n) for g,n in zip(gradient,numerical)) < 1e-6
step = [value - 0.2*deriv for value,deriv in zip(p,gradient)]
assert network_loss_gradient(x,labels,step)[0] < loss
print("OK")


## Interpretation

Annotate every tensor shape in the end-to-end exercise. Explain why finite differences can fail at ReLU kinks and why checking several coordinates is stronger than checking only a gradient norm.

**Your answer**
